# Preprocess DROID / Raw Videos to Masked-HWM Tokens (Kaggle)

This notebook downloads a novel robot dataset, starts with **DROID**, and converts RGB camera streams into the tokenized layout expected by `Masked-HWM/train.py`.

Default DROID source:

```text
gs://gresearch/robotics/droid_100
```

DROID provides an RLDS version for policy training and a raw MP4 version. For Kaggle, use `droid_100` first because the full RLDS dataset is ~1.7TB and the raw MP4 dataset is much larger.

Output layout:

```text
/kaggle/working/hwm_tokenized_dataset/
  train_v2.0/
    metadata.json
    metadata/metadata_0.json
    videos/video_0.bin
    segment_indices/segment_idx_0.bin
  val_v2.0/
    metadata.json
    metadata_0.json
    video_0.bin
    segment_idx_0.bin
```

Each 17-frame RGB clip is encoded with Cosmos DV8x8x8 into `(3, 32, 32)` int32 tokens, matching the current Masked-HWM Kaggle training config `T=3`.


In [1]:
# Settings: dataset link and conversion options.
from pathlib import Path

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

# DROID official small sample for debugging (~2GB). Use this first on Kaggle.
DATASET_NAME = "droid"
DATASET_LINK = "gs://gresearch/robotics/droid_100"
DOWNLOAD_DATASET = True
RAW_DOWNLOAD_DIR = KAGGLE_WORKING_ROOT / "raw_datasets"
RAW_DATASET_ROOT = RAW_DOWNLOAD_DIR / "droid_100"

# If you already added/pre-downloaded a dataset, set DOWNLOAD_DATASET=False and point RAW_DATASET_ROOT to it.
# RAW_DATASET_ROOT = Path("/kaggle/input/my-droid-rlds")

# DROID camera keys. Try wrist first for robot-hand view; exterior cameras are wider.
CAMERA_KEY_CANDIDATES = [
    "observation.exterior_image_1_left",
    "observation.exterior_image_2_left",
    "observation.wrist_image_left",
    "observation.image",
]

# Optional fallback: if your dataset is plain videos or frame folders instead of RLDS.
RAW_VIDEO_ROOTS = []

# Cosmos tokenizer files. The notebook searches this root and /kaggle/input for encoder.jit/decoder.jit.
COSMOS_INPUT_ROOT = Path("/kaggle/input/hw-model-2")

OUTPUT_ROOT = KAGGLE_WORKING_ROOT / "hwm_tokenized_dataset"
TRAIN_DIR = OUTPUT_ROOT / "train_v2.0"
VAL_DIR = OUTPUT_ROOT / "val_v2.0"

WINDOW_SIZE = 17
TARGET_SIZE = 256
FRAME_STRIDE = 1
VAL_RATIO = 0.05
RANDOM_SEED = 5
DATASET_HZ = 10

# Keep small for first run. Set to None after smoke test.
MAX_EPISODES = None
MAX_CLIPS_TOTAL = None

BATCH_CLIPS = 2
SHARD_SIZE_CLIPS = 4096

VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

print("Dataset link:", DATASET_LINK)
print("Raw dataset root:", RAW_DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)


Dataset link: gs://gresearch/robotics/droid_100
Raw dataset root: /kaggle/working/raw_datasets/droid_100
Output root: /kaggle/working/hwm_tokenized_dataset


In [2]:
# Download DROID dataset link into /kaggle/working.
import shutil
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None, check=True):
    print("\n$", " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


if DOWNLOAD_DATASET:
    RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    if RAW_DATASET_ROOT.exists() and any(RAW_DATASET_ROOT.iterdir()):
        print("Dataset already exists:", RAW_DATASET_ROOT)
    else:
        if DATASET_LINK.startswith("gs://"):
            if shutil.which("gsutil") is None:
                run([sys.executable, "-m", "pip", "install", "-q", "gsutil"])
            run(["gsutil", "-m", "cp", "-r", DATASET_LINK, str(RAW_DOWNLOAD_DIR)])
        else:
            # Generic URL/file fallback. For archives, upload/extract manually if needed.
            run(["wget", "-nc", "-P", str(RAW_DOWNLOAD_DIR), DATASET_LINK])
else:
    print("DOWNLOAD_DATASET=False; using existing RAW_DATASET_ROOT:", RAW_DATASET_ROOT)

print("Raw dataset tree sample:")
for p in sorted(RAW_DATASET_ROOT.rglob("*"))[:40]:
    print("-", p.relative_to(RAW_DATASET_ROOT))



$ gsutil -m cp -r gs://gresearch/robotics/droid_100 /kaggle/working/raw_datasets
Copying gs://gresearch/robotics/droid_100/1.0.0/dataset_info.json...
/ [0/33 files][    0.0 B/  2.0 GiB]   0% Done                                   
Copying gs://gresearch/robotics/droid_100/1.0.0/r2d2_faceblur-train.tfrecord-00000-of-00031...
/ [0/33 files][    0.0 B/  2.0 GiB]   0% Done                                   
Copying gs://gresearch/robotics/droid_100/1.0.0/features.json...
/ [0/33 files][    0.0 B/  2.0 GiB]   0% Done                                   
Copying gs://gresearch/robotics/droid_100/1.0.0/r2d2_faceblur-train.tfrecord-00001-of-00031...
Copying gs://gresearch/robotics/droid_100/1.0.0/r2d2_faceblur-train.tfrecord-00003-of-00031...
/ [0/33 files][    0.0 B/  2.0 GiB]   0% Done                                   
Copying gs://gresearch/robotics/droid_100/1.0.0/r2d2_faceblur-train.tfrecord-00014-of-00031...
/ [0/33 files][    0.0 B/  2.0 GiB]   0% Done                                   

In [3]:
# Install/import TensorFlow Datasets for DROID RLDS and Cosmos-Tokenizer for encoding.
# This mirrors the working Kaggle inference setup, but adds the encoder-side deps
# that are needed when importing NVIDIA/Cosmos-Tokenizer from source.
import importlib
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def ensure_import(package_name, pip_name=None):
    try:
        return importlib.import_module(package_name)
    except ModuleNotFoundError:
        run([sys.executable, "-m", "pip", "install", "-q", pip_name or package_name])
        importlib.invalidate_caches()
        return importlib.import_module(package_name)


# TensorFlow emits some harmless CUDA plugin warnings on Kaggle when PyTorch is also loaded.
tf = ensure_import("tensorflow", "tensorflow")
tfds = ensure_import("tensorflow_datasets", "tensorflow_datasets")

# Cosmos-Tokenizer imports these at module import time. Installing them explicitly
# avoids the `ModuleNotFoundError: mediapy` failure seen on fresh Kaggle sessions.
COSMOS_RUNTIME_DEPS = [
    ("mediapy", "mediapy"),
    ("einops", "einops"),
    ("omegaconf", "omegaconf"),
    ("loguru", "loguru"),
    ("safetensors", "safetensors"),
    ("huggingface_hub", "huggingface_hub"),
    ("imageio", "imageio"),
    ("cv2", "opencv-python-headless"),
]
for import_name, pip_name in COSMOS_RUNTIME_DEPS:
    ensure_import(import_name, pip_name)


def import_cosmos_tokenizer_from_source(source_root):
    source_root = Path(source_root)
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))
    importlib.invalidate_caches()
    try:
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer
    except ModuleNotFoundError as exc:
        missing = exc.name
        pip_name = {
            "cv2": "opencv-python-headless",
            "PIL": "pillow",
        }.get(missing, missing)
        print(f"Missing dependency while importing Cosmos-Tokenizer: {missing}. Installing {pip_name} and retrying...")
        run([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer


def find_cosmos_tokenizer_source_roots(*roots):
    source_roots = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for package_file in root.rglob("cosmos_tokenizer/video_lib.py"):
            source_roots.append(package_file.parents[1])
    return source_roots


def ensure_cosmos_tokenizer_package():
    try:
        from cosmos_tokenizer.video_lib import CausalVideoTokenizer
        return CausalVideoTokenizer
    except ModuleNotFoundError:
        pass

    for source_root in find_cosmos_tokenizer_source_roots(COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, KAGGLE_WORKING_ROOT):
        try:
            print("Using Cosmos-Tokenizer source from:", source_root)
            return import_cosmos_tokenizer_from_source(source_root)
        except ModuleNotFoundError:
            continue

    cosmos_repo = KAGGLE_WORKING_ROOT / "Cosmos-Tokenizer"
    if not cosmos_repo.exists():
        print("Cosmos-Tokenizer source not found; cloning NVIDIA/Cosmos-Tokenizer...")
        run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/Cosmos-Tokenizer.git", str(cosmos_repo)])

    for source_root in find_cosmos_tokenizer_source_roots(cosmos_repo):
        print("Using Cosmos-Tokenizer source from:", source_root)
        return import_cosmos_tokenizer_from_source(source_root)

    run([sys.executable, "-m", "pip", "install", "-e", str(cosmos_repo)])
    importlib.invalidate_caches()
    from cosmos_tokenizer.video_lib import CausalVideoTokenizer
    return CausalVideoTokenizer


def find_first_file(filename, roots):
    candidates = []
    for root in roots:
        root = Path(root)
        if root.exists():
            direct = root / filename
            if direct.exists():
                candidates.append(direct)
            candidates.extend(root.rglob(filename))
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        return candidate
    return None


CausalVideoTokenizer = ensure_cosmos_tokenizer_package()
ENCODER_JIT = find_first_file("encoder.jit", [COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, Path.cwd()])
DECODER_JIT = find_first_file("decoder.jit", [COSMOS_INPUT_ROOT, KAGGLE_INPUT_ROOT, Path.cwd()])

if ENCODER_JIT is None:
    raise FileNotFoundError("Could not find encoder.jit. Add Cosmos-0.1-Tokenizer-DV8x8x8 as Kaggle input.")
print("Encoder:", ENCODER_JIT)
print("Decoder:", DECODER_JIT)


2026-06-19 16:24:04.586713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781886244.877165      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781886244.940987      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781886245.437148      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781886245.437196      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781886245.437199      58 computation_placer.cc:177] computation placer alr


$ /usr/bin/python3 -m pip install -q mediapy


$ /usr/bin/python3 -m pip install -q loguru
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.6 MB/s eta 0:00:00

Cosmos-Tokenizer source not found; cloning NVIDIA/Cosmos-Tokenizer...

$ git clone --depth 1 https://github.com/NVIDIA/Cosmos-Tokenizer.git /kaggle/working/Cosmos-Tokenizer
Cloning into '/kaggle/working/Cosmos-Tokenizer'...

Using Cosmos-Tokenizer source from: /kaggle/working/Cosmos-Tokenizer
Encoder: /kaggle/input/datasets/angelminh/hw-model-2/Cosmos-0.1-Tokenizer-DV8x8x8/encoder.jit
Decoder: /kaggle/input/datasets/angelminh/hw-model-2/Cosmos-0.1-Tokenizer-DV8x8x8/decoder.jit


In [4]:
# Find DROID RLDS builder directories and inspect one episode.
import json
import random
from pathlib import Path


def find_tfds_builder_dirs(root):
    root = Path(root)
    return sorted(p.parent for p in root.rglob("dataset_info.json"))


def print_mapping_tree(node, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    if isinstance(node, dict):
        for k, v in node.items():
            name = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                print(name)
                print_mapping_tree(v, name, depth + 1, max_depth)
            else:
                shape = getattr(v, "shape", None)
                dtype = getattr(v, "dtype", None)
                print(f"{name}: shape={shape}, dtype={dtype}")


RLDS_BUILDER_DIRS = find_tfds_builder_dirs(RAW_DATASET_ROOT)
print("RLDS builder dirs:")
for d in RLDS_BUILDER_DIRS:
    print("-", d)

if not RLDS_BUILDER_DIRS:
    print("No RLDS builder dirs found. The notebook will fall back to scanning video files/frame folders.")
else:
    DROID_BUILDER_DIR = RLDS_BUILDER_DIRS[0]
    builder = tfds.builder_from_directory(str(DROID_BUILDER_DIR))
    print("Builder:", builder.name, builder.version)
    print("Splits:", builder.info.splits)

    split_name = "train" if "train" in builder.info.splits else list(builder.info.splits.keys())[0]
    sample_tfds = builder.as_dataset(split=split_name, shuffle_files=False).take(1)
    print("Element spec tree:")
    print_mapping_tree(sample_tfds.element_spec)

    sample_ep = next(iter(tfds.as_numpy(sample_tfds)))
    print("Top-level keys:", list(sample_ep.keys()))
    first_step = next(iter(sample_ep["steps"]))
    print("First step tree:")
    print_mapping_tree(first_step)


RLDS builder dirs:
- /kaggle/working/raw_datasets/droid_100/1.0.0
Builder: r2d2_faceblur 1.0.0
Splits: {'train': <SplitInfo num_examples=100, num_shards=31>}


I0000 00:00:1781886285.246629      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781886285.252819      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Element spec tree:
episode_metadata
episode_metadata.file_path: shape=(), dtype=<dtype: 'string'>
episode_metadata.recording_folderpath: shape=(), dtype=<dtype: 'string'>
steps: shape=None, dtype=None
Top-level keys: ['episode_metadata', 'steps']
First step tree:
action: shape=(7,), dtype=float64
action_dict
action_dict.cartesian_position: shape=(6,), dtype=float64
action_dict.cartesian_velocity: shape=(6,), dtype=float64
action_dict.gripper_position: shape=(1,), dtype=float64
action_dict.gripper_velocity: shape=(1,), dtype=float64
action_dict.joint_position: shape=(7,), dtype=float64
action_dict.joint_velocity: shape=(7,), dtype=float64
discount: shape=(), dtype=float32
is_first: shape=(), dtype=bool
is_last: shape=(), dtype=bool
is_terminal: shape=(), dtype=bool
language_instruction: shape=None, dtype=None
language_instruction_2: shape=None, dtype=None
language_instruction_3: shape=None, dtype=None
observation
observation.cartesian_position: shape=(6,), dtype=float64
observation.exte

In [5]:
# Discover sources: DROID RLDS episodes, or fallback videos/frame folders.
import random
from pathlib import Path


def get_split_names(builder):
    names = list(builder.info.splits.keys())
    if "train" in names:
        return ["train"]
    return names[:1]


def discover_rlds_episode_sources():
    if not RLDS_BUILDER_DIRS:
        return []
    sources = []
    builder_dir = RLDS_BUILDER_DIRS[0]
    builder = tfds.builder_from_directory(str(builder_dir))
    for split in get_split_names(builder):
        n = builder.info.splits[split].num_examples
        for episode_index in range(n):
            sources.append(("rlds", {"builder_dir": str(builder_dir), "split": split, "episode_index": episode_index}))
    return sources


def find_video_files(root):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)


def find_frame_dirs(root):
    root = Path(root)
    frame_dirs = []
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        imgs = [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) >= WINDOW_SIZE:
            frame_dirs.append(d)
    return sorted(frame_dirs)


sources = discover_rlds_episode_sources()
if not sources:
    scan_roots = RAW_VIDEO_ROOTS or [RAW_DATASET_ROOT]
    for root in scan_roots:
        sources.extend(("video", p) for p in find_video_files(root))
        sources.extend(("frames", p) for p in find_frame_dirs(root))

random.Random(RANDOM_SEED).shuffle(sources)
if MAX_EPISODES is not None:
    sources = sources[:MAX_EPISODES]

print("Found sources:", len(sources))
for item in sources[:10]:
    print(item)

if not sources:
    raise RuntimeError("No RLDS episodes, video files, or frame folders found.")


Found sources: 100
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 91})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 2})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 30})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 33})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 58})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 68})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 85})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split': 'train', 'episode_index': 7})
('rlds', {'builder_dir': '/kaggle/working/raw_datasets/droid_100/1.0.0', 'split

In [6]:
# Frame loading and preprocessing helpers.
import cv2
import numpy as np
import torch
from PIL import Image
from collections.abc import Mapping

_BUILDER_CACHE = {}
_DATASET_CACHE = {}
_EPISODE_FRAME_CACHE = {}
_PRINTED_CAMERA_KEYS = set()


def resize_center_crop_rgb(frame_rgb, target_size=TARGET_SIZE):
    img = Image.fromarray(frame_rgb.astype(np.uint8)).convert("RGB")
    w, h = img.size
    scale = target_size / min(w, h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    img = img.resize((new_w, new_h), Image.BICUBIC)
    left = (new_w - target_size) // 2
    top = (new_h - target_size) // 2
    img = img.crop((left, top, left + target_size, top + target_size))
    return np.asarray(img, dtype=np.uint8)


def get_nested_value(node, dotted_key):
    cur = node
    for part in dotted_key.split("."):
        if not isinstance(cur, Mapping) or part not in cur:
            return None
        cur = cur[part]
    return cur


def normalize_frame_array(frame):
    frame = np.asarray(frame)
    if frame.ndim == 4 and frame.shape[0] == 1:
        frame = frame[0]
    if frame.ndim == 2:
        frame = np.repeat(frame[..., None], 3, axis=-1)
    if frame.ndim != 3:
        return None
    if frame.shape[0] in (1, 3) and frame.shape[-1] not in (1, 3):
        frame = np.moveaxis(frame, 0, -1)
    if frame.shape[-1] == 1:
        frame = np.repeat(frame, 3, axis=-1)
    if frame.shape[-1] != 3:
        return None
    if frame.dtype != np.uint8:
        if frame.max(initial=0) <= 1.0:
            frame = frame * 255.0
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    return frame


def find_first_image_in_mapping(node, prefix=""):
    if isinstance(node, Mapping):
        for k, v in node.items():
            child_prefix = f"{prefix}.{k}" if prefix else str(k)
            found = find_first_image_in_mapping(v, child_prefix)
            if found is not None:
                return found
        return None
    frame = normalize_frame_array(node)
    if frame is not None:
        return frame, prefix
    return None


def pick_step_frame(step):
    # DROID RLDS stores images inside each step, usually under step["observation"].
    for key in CAMERA_KEY_CANDIDATES:
        frame = get_nested_value(step, key)
        frame = normalize_frame_array(frame) if frame is not None else None
        if frame is not None:
            return frame, key

    # Generic fallback: first HWC-looking image anywhere in the step mapping.
    found = find_first_image_in_mapping(step)
    if found is not None:
        return found
    raise KeyError(f"Could not find an image camera in a DROID step. Tried: {CAMERA_KEY_CANDIDATES}")


def get_dataset_iterable(builder_dir, split):
    if builder_dir not in _BUILDER_CACHE:
        _BUILDER_CACHE[builder_dir] = tfds.builder_from_directory(builder_dir)
    builder = _BUILDER_CACHE[builder_dir]
    key = (builder_dir, split)
    if key not in _DATASET_CACHE:
        _DATASET_CACHE[key] = tfds.as_numpy(builder.as_dataset(split=split, shuffle_files=False))
    return _DATASET_CACHE[key]


def load_rlds_episode_frames(source_info):
    builder_dir = source_info["builder_dir"]
    split = source_info["split"]
    episode_index = source_info["episode_index"]
    cache_key = (builder_dir, split, episode_index, FRAME_STRIDE, TARGET_SIZE)
    if cache_key in _EPISODE_FRAME_CACHE:
        return _EPISODE_FRAME_CACHE[cache_key]

    dataset = get_dataset_iterable(builder_dir, split)
    for i, episode in enumerate(dataset):
        if i != episode_index:
            continue

        if "steps" not in episode:
            raise KeyError(f"Episode has no 'steps'. Top-level keys: {list(episode.keys())}")

        frames = []
        camera_key = None
        for step_i, step in enumerate(episode["steps"]):
            if step_i % FRAME_STRIDE != 0:
                continue
            frame, used_key = pick_step_frame(step)
            camera_key = camera_key or used_key
            frames.append(resize_center_crop_rgb(frame))

        if not frames:
            raise RuntimeError(f"Episode {episode_index} has no usable frames after FRAME_STRIDE={FRAME_STRIDE}")

        print_once_key = (builder_dir, split, episode_index)
        if print_once_key not in _PRINTED_CAMERA_KEYS:
            print(f"Episode {episode_index} camera: {camera_key}, frames: {len(frames)}")
            _PRINTED_CAMERA_KEYS.add(print_once_key)

        _EPISODE_FRAME_CACHE[cache_key] = frames
        return frames

    raise IndexError(f"Episode index out of range: {episode_index}")


def load_video_frames(path):
    cap = cv2.VideoCapture(str(path))
    frames = []
    i = 0
    ok, frame_bgr = cap.read()
    while ok:
        if i % FRAME_STRIDE == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append(resize_center_crop_rgb(frame_rgb))
        i += 1
        ok, frame_bgr = cap.read()
    cap.release()
    return frames


def load_frame_dir_frames(path):
    image_paths = sorted(p for p in Path(path).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    frames = []
    for i, image_path in enumerate(image_paths):
        if i % FRAME_STRIDE != 0:
            continue
        frame_rgb = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.uint8)
        frames.append(resize_center_crop_rgb(frame_rgb))
    return frames


def load_source_frames(kind, payload):
    if kind == "rlds":
        return load_rlds_episode_frames(payload)
    if kind == "video":
        return load_video_frames(payload)
    if kind == "frames":
        return load_frame_dir_frames(payload)
    raise ValueError(f"Unknown source kind: {kind}")


def iter_nonoverlap_windows(frames):
    usable = (len(frames) // WINDOW_SIZE) * WINDOW_SIZE
    for start in range(0, usable, WINDOW_SIZE):
        yield np.stack(frames[start:start + WINDOW_SIZE], axis=0)


def clips_to_cosmos_tensor(clips):
    arr = np.stack(clips, axis=0)
    x = torch.from_numpy(arr).permute(0, 4, 1, 2, 3).float()
    x = x / 127.5 - 1.0
    return x.to(device="cuda", dtype=torch.bfloat16)


In [7]:
# Convert discovered sources to Masked-HWM tokenized train/val layout.
import json
import math
import shutil
from tqdm.auto import tqdm

encoder = CausalVideoTokenizer(checkpoint_enc=str(ENCODER_JIT))


def prepare_output_dirs():
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
    (TRAIN_DIR / "metadata").mkdir(parents=True, exist_ok=True)
    (TRAIN_DIR / "videos").mkdir(parents=True, exist_ok=True)
    (TRAIN_DIR / "segment_indices").mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)


def write_split(split_name, split_sources, output_dir, is_val=False):
    shard_id = 0
    shard_tokens = []
    shard_segment_ids = []
    shard_frames = 0
    segment_id = 0
    total_clips = 0
    total_frames = 0
    shard_metas = []

    def flush_shard():
        nonlocal shard_id, shard_tokens, shard_segment_ids, shard_frames
        if not shard_tokens:
            return
        tokens_arr = np.concatenate(shard_tokens, axis=0).astype(np.int32)
        segments_arr = np.asarray(shard_segment_ids, dtype=np.int32)

        if is_val:
            tokens_arr.tofile(output_dir / f"video_{shard_id}.bin")
            segments_arr.tofile(output_dir / f"segment_idx_{shard_id}.bin")
            (output_dir / f"metadata_{shard_id}.json").write_text(json.dumps({"shard_num_frames": shard_frames}))
        else:
            tokens_arr.tofile(output_dir / "videos" / f"video_{shard_id}.bin")
            segments_arr.tofile(output_dir / "segment_indices" / f"segment_idx_{shard_id}.bin")
            (output_dir / "metadata" / f"metadata_{shard_id}.json").write_text(json.dumps({"shard_num_frames": shard_frames}))

        shard_metas.append({"shard": shard_id, "clips": int(tokens_arr.shape[0]), "frames": int(shard_frames)})
        print(f"Wrote {split_name} shard {shard_id}: clips={tokens_arr.shape[0]}, frames={shard_frames}")
        shard_id += 1
        shard_tokens = []
        shard_segment_ids = []
        shard_frames = 0

    for kind, payload in tqdm(split_sources, desc=f"Encoding {split_name}"):
        if MAX_CLIPS_TOTAL is not None and total_clips >= MAX_CLIPS_TOTAL:
            break

        try:
            frames = load_source_frames(kind, payload)
        except Exception as exc:
            print("Skipping unreadable source:", kind, payload, repr(exc))
            continue

        windows = list(iter_nonoverlap_windows(frames))
        if len(windows) < 2:
            continue

        for batch_start in range(0, len(windows), BATCH_CLIPS):
            if MAX_CLIPS_TOTAL is not None and total_clips >= MAX_CLIPS_TOTAL:
                break
            batch_windows = windows[batch_start:batch_start + BATCH_CLIPS]
            if MAX_CLIPS_TOTAL is not None:
                batch_windows = batch_windows[:MAX_CLIPS_TOTAL - total_clips]
            if not batch_windows:
                break

            x = clips_to_cosmos_tensor(batch_windows)
            with torch.no_grad():
                indices, _ = encoder.encode(x)
            tokens = indices.detach().cpu().numpy().astype(np.int32)
            if tokens.shape[1:] != (3, 32, 32):
                raise ValueError(f"Unexpected token shape {tokens.shape}; expected (B,3,32,32). Check tokenizer variant/input size.")

            shard_tokens.append(tokens)
            frames_in_batch = len(batch_windows) * WINDOW_SIZE
            shard_segment_ids.extend([segment_id] * frames_in_batch)
            shard_frames += frames_in_batch
            total_clips += len(batch_windows)
            total_frames += frames_in_batch

            if sum(t.shape[0] for t in shard_tokens) >= SHARD_SIZE_CLIPS:
                flush_shard()

        segment_id += 1

    flush_shard()
    (output_dir / "metadata.json").write_text(json.dumps({"num_shards": shard_id, "hz": DATASET_HZ}, indent=2))
    (output_dir / "preprocess_summary.json").write_text(json.dumps({
        "split": split_name,
        "num_sources": len(split_sources),
        "total_clips": total_clips,
        "total_frames": total_frames,
        "window_size": WINDOW_SIZE,
        "token_shape": [3, 32, 32],
        "shards": shard_metas,
    }, indent=2))
    return total_clips, total_frames, shard_id


prepare_output_dirs()
val_count = max(1, int(len(sources) * VAL_RATIO)) if len(sources) > 1 else 0
val_sources = sources[:val_count]
train_sources = sources[val_count:]
if not train_sources and val_sources:
    train_sources, val_sources = val_sources, []

print("Train sources:", len(train_sources))
print("Val sources:", len(val_sources))

train_stats = write_split("train", train_sources, TRAIN_DIR, is_val=False)
val_stats = write_split("val", val_sources, VAL_DIR, is_val=True) if val_sources else (0, 0, 0)
print("Train stats:", train_stats)
print("Val stats:", val_stats)
print("Output root:", OUTPUT_ROOT)


Train sources: 95
Val sources: 5


Encoding train:   0%|          | 0/95 [00:00<?, ?it/s]

Episode 68 camera: observation.exterior_image_1_left, frames: 211
Episode 85 camera: observation.exterior_image_1_left, frames: 130
Episode 7 camera: observation.exterior_image_1_left, frames: 596
Episode 74 camera: observation.exterior_image_1_left, frames: 167
Episode 29 camera: observation.exterior_image_1_left, frames: 227
Episode 50 camera: observation.exterior_image_1_left, frames: 1347
Episode 53 camera: observation.exterior_image_1_left, frames: 388
Episode 87 camera: observation.exterior_image_1_left, frames: 573
Episode 39 camera: observation.exterior_image_1_left, frames: 684
Episode 98 camera: observation.exterior_image_1_left, frames: 1120
Episode 92 camera: observation.exterior_image_1_left, frames: 125
Episode 84 camera: observation.exterior_image_1_left, frames: 1627
Episode 61 camera: observation.exterior_image_1_left, frames: 50
Episode 78 camera: observation.exterior_image_1_left, frames: 323
Episode 8 camera: observation.exterior_image_1_left, frames: 114
Episode 81

Encoding val:   0%|          | 0/5 [00:00<?, ?it/s]

Episode 91 camera: observation.exterior_image_1_left, frames: 100
Episode 2 camera: observation.exterior_image_1_left, frames: 142
Episode 30 camera: observation.exterior_image_1_left, frames: 361
Episode 33 camera: observation.exterior_image_1_left, frames: 281
Episode 58 camera: observation.exterior_image_1_left, frames: 213
Wrote val shard 0: clips=62, frames=1054
Train stats: (1784, 30328, 1)
Val stats: (62, 1054, 1)
Output root: /kaggle/working/hwm_tokenized_dataset


In [8]:
# Validate output files and optionally decode a sample clip.
import json
import math
import numpy as np
import torch

print("Output tree sample:")
for p in sorted(OUTPUT_ROOT.rglob("*"))[:80]:
    print("-", p.relative_to(OUTPUT_ROOT))

train_meta = json.loads((TRAIN_DIR / "metadata.json").read_text())
assert train_meta["num_shards"] > 0, "No train shards were written."
shard_meta = json.loads((TRAIN_DIR / "metadata" / "metadata_0.json").read_text())
num_chunks = math.ceil(shard_meta["shard_num_frames"] / WINDOW_SIZE)
tokens = np.memmap(TRAIN_DIR / "videos" / "video_0.bin", dtype=np.int32, mode="r", shape=(num_chunks, 3, 32, 32))
segments = np.memmap(TRAIN_DIR / "segment_indices" / "segment_idx_0.bin", dtype=np.int32, mode="r", shape=(shard_meta["shard_num_frames"],))
print("First train shard token shape:", tokens.shape)
print("First train shard segment shape:", segments.shape)

if DECODER_JIT is not None:
    decoder = CausalVideoTokenizer(checkpoint_dec=str(DECODER_JIT))
    sample = torch.from_numpy(np.asarray(tokens[0:1]).copy()).long().cuda()
    with torch.no_grad():
        decoded = decoder.decode(sample).detach().float().cpu()
    print("Decoded sample shape:", tuple(decoded.shape))
else:
    print("Skipping decode validation because decoder.jit was not found.")


Output tree sample:
- train_v2.0
- train_v2.0/metadata
- train_v2.0/metadata/metadata_0.json
- train_v2.0/metadata.json
- train_v2.0/preprocess_summary.json
- train_v2.0/segment_indices
- train_v2.0/segment_indices/segment_idx_0.bin
- train_v2.0/videos
- train_v2.0/videos/video_0.bin
- val_v2.0
- val_v2.0/metadata.json
- val_v2.0/metadata_0.json
- val_v2.0/preprocess_summary.json
- val_v2.0/segment_idx_0.bin
- val_v2.0/video_0.bin
First train shard token shape: (1784, 3, 32, 32)
First train shard segment shape: (30328,)
Decoded sample shape: (1, 3, 17, 256, 256)


In [9]:
# Compare the generated DROID token dataset against the original 1X/world-model layout.
# Run this after the validate cell. It checks the exact files RawTokenDataset expects.
import json
import math
from pathlib import Path

import numpy as np

REFERENCE_1X_ROOT = None  # Optional manual override, e.g. Path("/kaggle/input/.../world-model")
GENERATED_DROID_ROOT = OUTPUT_ROOT


def status_line(ok, label, detail=""):
    tag = "OK" if ok else "FAIL"
    msg = f"[{tag}] {label}"
    if detail:
        msg += f": {detail}"
    print(msg)
    return ok


def find_dataset_roots(search_root=KAGGLE_INPUT_ROOT):
    roots = []
    search_root = Path(search_root)
    if not search_root.exists():
        return roots
    for meta in search_root.rglob("train_v2.0/metadata.json"):
        root = meta.parents[1]
        if (root / "val_v2.0" / "metadata.json").exists():
            roots.append(root)
    # Prefer roots that look like the 1X/world-model dataset shown in the Kaggle input panel.
    roots = sorted(set(roots), key=lambda x: ("world-model" not in str(x).lower(), len(str(x))))
    return roots


def auto_reference_root():
    if REFERENCE_1X_ROOT is not None:
        return Path(REFERENCE_1X_ROOT)
    roots = find_dataset_roots()
    generated = Path(GENERATED_DROID_ROOT).resolve()
    for root in roots:
        try:
            if root.resolve() == generated:
                continue
        except FileNotFoundError:
            pass
        return root
    raise FileNotFoundError(
        "Could not auto-detect a reference 1X dataset under /kaggle/input. "
        "Set REFERENCE_1X_ROOT manually to the folder containing train_v2.0 and val_v2.0."
    )


def split_paths(root, split):
    root = Path(root)
    split_dir = root / f"{split}_v2.0"
    is_val = split == "val"
    return {
        "split_dir": split_dir,
        "metadata": split_dir / "metadata.json",
        "shard_meta": (split_dir / "metadata_0.json") if is_val else (split_dir / "metadata" / "metadata_0.json"),
        "video": (split_dir / "video_0.bin") if is_val else (split_dir / "videos" / "video_0.bin"),
        "segment": (split_dir / "segment_idx_0.bin") if is_val else (split_dir / "segment_indices" / "segment_idx_0.bin"),
        "robot_states": None if is_val else (split_dir / "robot_states" / "states_0.bin"),
    }


def inspect_split(root, split, window_size=WINDOW_SIZE):
    paths = split_paths(root, split)
    result = {
        "root": str(root),
        "split": split,
        "ok": True,
        "paths": {k: str(v) if v is not None else None for k, v in paths.items()},
    }

    required = ["split_dir", "metadata", "shard_meta", "video", "segment"]
    for key in required:
        exists = paths[key].exists()
        result[f"{key}_exists"] = exists
        result["ok"] &= exists
    if not result["ok"]:
        return result

    meta = json.loads(paths["metadata"].read_text())
    shard_meta = json.loads(paths["shard_meta"].read_text())
    frames = int(shard_meta["shard_num_frames"])
    chunks = math.ceil(frames / window_size)
    video_bytes = paths["video"].stat().st_size
    segment_bytes = paths["segment"].stat().st_size
    tokens_per_chunk = video_bytes // max(1, chunks * np.dtype(np.int32).itemsize)
    token_shape_ok = tokens_per_chunk == 3 * 32 * 32
    video_size_ok = video_bytes == chunks * 3 * 32 * 32 * np.dtype(np.int32).itemsize
    segment_size_ok = segment_bytes == frames * np.dtype(np.int32).itemsize

    result.update({
        "num_shards": int(meta.get("num_shards", -1)),
        "hz": meta.get("hz"),
        "shard_num_frames": frames,
        "chunks": chunks,
        "video_bytes": video_bytes,
        "segment_bytes": segment_bytes,
        "tokens_per_chunk": tokens_per_chunk,
        "token_shape": [3, 32, 32] if token_shape_ok else [tokens_per_chunk],
        "token_shape_ok": token_shape_ok,
        "video_size_ok": video_size_ok,
        "segment_size_ok": segment_size_ok,
        "robot_states_exists": bool(paths["robot_states"] and paths["robot_states"].exists()),
    })
    result["ok"] &= result["num_shards"] > 0 and token_shape_ok and video_size_ok and segment_size_ok

    if result["ok"]:
        tokens = np.memmap(paths["video"], dtype=np.int32, mode="r", shape=(chunks, 3, 32, 32))
        segments = np.memmap(paths["segment"], dtype=np.int32, mode="r", shape=(frames,))
        effective_chunks = chunks - (1 if frames % window_size != 0 else 0)
        valid_examples = 0
        for i in range(max(0, effective_chunks - 1)):
            idx1 = i * window_size
            idx2 = (i + 1) * window_size - 1
            if idx2 < len(segments) and segments[idx1] == segments[idx2]:
                valid_examples += 1
        result.update({
            "dtype": str(tokens.dtype),
            "first_token_min": int(np.asarray(tokens[0]).min()) if chunks else None,
            "first_token_max": int(np.asarray(tokens[0]).max()) if chunks else None,
            "unique_segments_first_64": int(len(np.unique(np.asarray(segments[:min(len(segments), 64)])))),
            "raw_token_dataset_valid_examples_estimate": valid_examples,
        })
    return result


def print_split_report(name, report):
    print(f"\n{name} {report['split']} split")
    status_line(report.get("split_dir_exists", False), "split directory", report["paths"]["split_dir"])
    status_line(report.get("metadata_exists", False), "metadata.json")
    status_line(report.get("shard_meta_exists", False), "first shard metadata")
    status_line(report.get("video_exists", False), "first video bin")
    status_line(report.get("segment_exists", False), "first segment bin")
    if not all(report.get(f"{k}_exists", False) for k in ["split_dir", "metadata", "shard_meta", "video", "segment"]):
        return
    status_line(report["num_shards"] > 0, "num_shards", str(report["num_shards"]))
    status_line(report["token_shape_ok"], "token shape", str(report["token_shape"]))
    status_line(report["video_size_ok"], "video file size matches metadata", f"chunks={report['chunks']}, bytes={report['video_bytes']}")
    status_line(report["segment_size_ok"], "segment file size matches metadata", f"frames={report['shard_num_frames']}, bytes={report['segment_bytes']}")
    print("hz:", report.get("hz"))
    print("robot_states exists:", report.get("robot_states_exists"), "(required only when WITH_ACT=True)")
    print("RawTokenDataset valid examples estimate:", report.get("raw_token_dataset_valid_examples_estimate"))
    print("first token min/max:", report.get("first_token_min"), report.get("first_token_max"))


reference_root = auto_reference_root()
generated_root = Path(GENERATED_DROID_ROOT)
print("Reference 1X root:", reference_root)
print("Generated DROID root:", generated_root)

reports = {}
for split in ["train", "val"]:
    reports[("1x", split)] = inspect_split(reference_root, split)
    reports[("droid", split)] = inspect_split(generated_root, split)
    print_split_report("1X reference", reports[("1x", split)])
    print_split_report("DROID generated", reports[("droid", split)])

print("\nStructure comparison summary")
for split in ["train", "val"]:
    ref = reports[("1x", split)]
    gen = reports[("droid", split)]
    comparable = ref["ok"] and gen["ok"]
    same_required_layout = all(
        ref.get(key) == gen.get(key)
        for key in ["token_shape_ok", "video_size_ok", "segment_size_ok"]
    )
    status_line(comparable and same_required_layout, f"{split}_v2.0 required layout matches")

print("\nTraining readiness")
train_ok = reports[("droid", "train")]["ok"] and reports[("droid", "train")].get("raw_token_dataset_valid_examples_estimate", 0) > 0
val_ok = reports[("droid", "val")]["ok"] and reports[("droid", "val")].get("raw_token_dataset_valid_examples_estimate", 0) > 0
status_line(train_ok, "DROID train split can be read by RawTokenDataset-style loader")
status_line(val_ok, "DROID val split can be read by RawTokenDataset-style loader")
print("Use WITH_ACT=False for DROID unless you add robot_states/states_*.bin with the 1X 25-dim action format.")


Reference 1X root: /kaggle/input/datasets/lnk135/world-model-dataset/world-model
Generated DROID root: /kaggle/working/hwm_tokenized_dataset

1X reference train split
[OK] split directory: /kaggle/input/datasets/lnk135/world-model-dataset/world-model/train_v2.0
[OK] metadata.json
[OK] first shard metadata
[OK] first video bin
[OK] first segment bin
[OK] num_shards: 100
[OK] token shape: [3, 32, 32]
[OK] video file size matches metadata: chunks=6621, bytes=81358848
[OK] segment file size matches metadata: frames=112542, bytes=450168
hz: 30
robot_states exists: True (required only when WITH_ACT=True)
RawTokenDataset valid examples estimate: 6309
first token min/max: 124 63843

DROID generated train split
[OK] split directory: /kaggle/working/hwm_tokenized_dataset/train_v2.0
[OK] metadata.json
[OK] first shard metadata
[OK] first video bin
[OK] first segment bin
[OK] num_shards: 1
[OK] token shape: [3, 32, 32]
[OK] video file size matches metadata: chunks=1784, bytes=21921792
[OK] segment

## Use This Output for Training

After conversion succeeds, create a Kaggle Dataset from:

```text
/kaggle/working/hwm_tokenized_dataset
```

Then in the training notebook, set:

```python
KAGGLE_DATASET_ROOT = Path("/kaggle/input/<your-tokenized-dataset-slug>/hwm_tokenized_dataset")
WITH_ACT = False
```

Recommended first training config for DROID:

```python
MAX_TRAIN_STEPS = 1000
T = 3
num_prompt_frames = 2
```

Scale up after sanity-check inference shows prompt/ground-truth decode correctly.

Sources used for the default DROID download path: official DROID docs list `gs://gresearch/robotics/droid_100` as the 100-episode RLDS debug dataset and describe the full RLDS/raw dataset options.


In [10]:
# Publish /kaggle/working/hwm_tokenized_dataset as a new private Kaggle Dataset.
# Before running, add KAGGLE_USERNAME and KAGGLE_KEY under Add-ons -> Secrets.
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

PUBLISH_ROOT = Path(OUTPUT_ROOT)  # Upload only tokenized train_v2.0 + val_v2.0, not raw_datasets.
KAGGLE_DATASET_SLUG = "droid-hwm-tokenized"
KAGGLE_DATASET_TITLE = "DROID Masked-HWM Cosmos Tokens"
KAGGLE_DATASET_LICENSE = "CC-BY-4.0"
MAKE_PRIVATE = True


def require_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(
            f"Missing Kaggle secret {name}. Add it under Add-ons -> Secrets and enable it for this notebook."
        ) from exc


if not PUBLISH_ROOT.exists():
    raise FileNotFoundError(f"Tokenized output does not exist: {PUBLISH_ROOT}")
for required in ["train_v2.0/metadata.json", "val_v2.0/metadata.json"]:
    if not (PUBLISH_ROOT / required).exists():
        raise FileNotFoundError(f"Missing required output file: {PUBLISH_ROOT / required}")

username = require_secret("KAGGLE_USERNAME").strip()
api_key = require_secret("KAGGLE_KEY").strip()
if not re.fullmatch(r"[a-z0-9-]+", KAGGLE_DATASET_SLUG):
    raise ValueError("KAGGLE_DATASET_SLUG must contain only lowercase letters, numbers, and hyphens.")

# Kaggle CLI accepts credentials through environment variables, so no kaggle.json is written.
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = api_key

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"],
    check=True,
)

metadata = {
    "id": f"{username}/{KAGGLE_DATASET_SLUG}",
    "title": KAGGLE_DATASET_TITLE,
    "licenses": [{"name": KAGGLE_DATASET_LICENSE}],
    "isPrivate": MAKE_PRIVATE,
}
metadata_path = PUBLISH_ROOT / "dataset-metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

files = [p for p in PUBLISH_ROOT.rglob("*") if p.is_file()]
total_bytes = sum(p.stat().st_size for p in files)
print("Publishing root:", PUBLISH_ROOT)
print("Dataset id:", metadata["id"])
print("Files:", len(files))
print(f"Total size: {total_bytes / (1024 ** 2):.2f} MiB")
print("Private:", MAKE_PRIVATE)

cmd = [
    "kaggle", "datasets", "create",
    "-p", str(PUBLISH_ROOT),
    "--dir-mode", "zip",
]
print("$", " ".join(cmd))
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    if "already exists" in result.stdout.lower():
        raise RuntimeError(
            "This dataset slug already exists. Change KAGGLE_DATASET_SLUG for a new dataset, "
            "or use `kaggle datasets version -p <folder> -m <message> --dir-mode zip` to update it."
        )
    raise RuntimeError(f"Kaggle dataset creation failed with exit code {result.returncode}.")

print("Created Kaggle Dataset:", f"https://www.kaggle.com/datasets/{username}/{KAGGLE_DATASET_SLUG}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 15.4 MB/s eta 0:00:00
Publishing root: /kaggle/working/hwm_tokenized_dataset
Dataset id: angelminh/droid-hwm-tokenized
Files: 11
Total size: 21.75 MiB
Private: True
$ kaggle datasets create -p /kaggle/working/hwm_tokenized_dataset --dir-mode zip
Starting upload for file val_v2.0.zip

  0%|          | 0.00/384k [00:00<?, ?B/s]
100%|██████████| 384k/384k [00:00<00:00, 1.36MB/s]
Upload successful: val_v2.0.zip (384KB)
Starting upload for file train_v2.0.zip

  0%|          | 0.00/10.3M [00:00<?, ?B/s]
100%|██████████| 10.3M/10.3M [00:00<00:00, 31.0MB/s]
Upload successful: train_v2.0.zip (10MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/angelminh/droid-hwm-tokenized

Created Kaggle Dataset: https://www.kaggle.com/datasets/angelminh/droid-hwm-tokenized


## BridgeData V2 preprocessing extension

The DROID pipeline above is left untouched. This optional section reuses the same Cosmos tokenizer, frame preprocessing, and 1X-compatible writer, but switches the input/output paths to BridgeData V2. Keep `RUN_BRIDGEDATA_V2_PREPROCESS = False` until the Bridge source path is configured.

In [ ]:
# Install/probe BridgeData V2 TFDS support.
# The official TFDS builder is named "bridge" and is very large (~387 GB), so this cell installs
# support and prints dataset metadata without downloading the full dataset unless you explicitly allow it.
import importlib
import subprocess
import sys
from pathlib import Path

INSTALL_BRIDGE_TFDS_SUPPORT = True
PROBE_BRIDGE_TFDS_BUILDER = True

if INSTALL_BRIDGE_TFDS_SUPPORT:
    # Kaggle images can ship an older TFDS build that does not include robotics.rtx.Bridge.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "tensorflow-datasets"], check=True)
    importlib.invalidate_caches()
    tfds = importlib.import_module("tensorflow_datasets")
    print("tensorflow_datasets:", tfds.__version__)

if PROBE_BRIDGE_TFDS_BUILDER:
    if globals().get("BRIDGE_USE_GCS_STREAMING", True):
        print("Recommended Bridge source: gs://gresearch/robotics/bridge/0.1.0/ (streaming, no local full download)")
        print("Run the Bridge settings/discovery cells next; they will use tfds.builder_from_directory on GCS.")
    bridge_name = globals().get("BRIDGE_TFDS_NAME", None)
    if bridge_name is None:
        bridge_name = "bridge"
    bridge_data_dir = globals().get("BRIDGE_TFDS_DATA_DIR", Path("/kaggle/working/raw_datasets/tfds"))
    try:
        bridge_builder = tfds.builder(bridge_name, data_dir=str(bridge_data_dir))
    except Exception as exc:
        print("Could not build TFDS dataset", bridge_name, "with installed tensorflow-datasets:", repr(exc))
        print("Trying tfds-nightly, which often has newer robotics dataset builders...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "tfds-nightly"], check=True)
        importlib.invalidate_caches()
        tfds = importlib.import_module("tensorflow_datasets")
        bridge_builder = tfds.builder(bridge_name, data_dir=str(bridge_data_dir))

    print("Bridge TFDS builder:", bridge_builder.name, bridge_builder.version)
    print("Builder data dir:", bridge_builder.data_dir)
    print("Already downloaded:", Path(bridge_builder.data_dir).exists())
    print("Expected splits after download:", bridge_builder.info.splits)
    print("Important: full Bridge TFDS is huge. Keep BRIDGE_ALLOW_TFDS_DOWNLOAD=False on Kaggle unless you have enough disk.")

In [ ]:
# BridgeData V2 settings.
# This section is intentionally disabled by default so the working DROID pipeline above stays safe.
from pathlib import Path

RUN_BRIDGEDATA_V2_PREPROCESS = False

BRIDGE_DATASET_NAME = "bridge_v2"

# Recommended on Kaggle: stream BridgeData V2 directly from the public Open X-Embodiment GCS builder.
# This avoids downloading the full ~387 GiB dataset into /kaggle/working. Internet must be enabled.
BRIDGE_USE_GCS_STREAMING = True
BRIDGE_GCS_BUILDER_DIR = "gs://gresearch/robotics/bridge/0.1.0/"

# Option A: Add a BridgeData V2 RLDS / TFDS export as a Kaggle input, then point here.
# The folder should contain a TFDS builder directory with dataset_info.json somewhere inside.
BRIDGE_RAW_DATASET_ROOT = Path("/kaggle/input/bridgedata-v2")
BRIDGE_SCAN_ALL_KAGGLE_INPUTS = False  # Set True only if you want auto-search across every mounted Kaggle input.

# Option B: Download/copy from a public GCS mirror into /kaggle/working.
# Set BRIDGE_DATASET_LINK to a gs://... folder and enable BRIDGE_DOWNLOAD_DATASET.
BRIDGE_DOWNLOAD_DATASET = False
BRIDGE_DATASET_LINK = None  # example: "gs://path/to/bridge_dataset"
BRIDGE_RAW_DOWNLOAD_DIR = KAGGLE_WORKING_ROOT / "raw_datasets"
BRIDGE_DOWNLOAD_TARGET_ROOT = BRIDGE_RAW_DOWNLOAD_DIR / "bridge_v2"

# Option C: If TFDS can build it by name in the Kaggle session, set the exact builder name.
# Different mirrors expose Bridge under different names, so this is explicit rather than guessed.
BRIDGE_TFDS_NAME = None  # Local TFDS builder name. Use only if you really want local download/build.
BRIDGE_TFDS_DATA_DIR = BRIDGE_RAW_DOWNLOAD_DIR / "tfds"
BRIDGE_ALLOW_TFDS_DOWNLOAD = False  # Full Bridge TFDS is ~387 GB; keep False on normal Kaggle sessions.

# Bridge camera names vary by release/mirror. The first usable image key wins.
BRIDGE_CAMERA_KEY_CANDIDATES = [
    "observation.image_0",
    "observation.image_1",
    "observation.image_2",
    "observation.image",
    "observation.images0",
    "observation.images1",
    "observation.images2",
    "observation.wrist_image",
    "observation.wrist_image_left",
    "observation.exterior_image_1_left",
    "image_0",
    "image_1",
    "image_2",
    "image",
    "images0",
    "images1",
    "images2",
]

# Fallback if the Bridge source is plain videos or frame folders instead of RLDS/TFDS.
BRIDGE_RAW_VIDEO_ROOTS = []

# Keep Bridge output separate from DROID output.
BRIDGE_OUTPUT_ROOT = KAGGLE_WORKING_ROOT / "bridge_hwm_tokenized_dataset"
BRIDGE_TRAIN_DIR = BRIDGE_OUTPUT_ROOT / "train_v2.0"
BRIDGE_VAL_DIR = BRIDGE_OUTPUT_ROOT / "val_v2.0"

BRIDGE_WINDOW_SIZE = WINDOW_SIZE
BRIDGE_TARGET_SIZE = TARGET_SIZE
BRIDGE_FRAME_STRIDE = 1
BRIDGE_VAL_RATIO = 0.05
BRIDGE_RANDOM_SEED = RANDOM_SEED
BRIDGE_DATASET_HZ = 10

# Use a smoke limit first, then set both to None for the full Bridge conversion.
BRIDGE_MAX_EPISODES = 200
BRIDGE_MAX_CLIPS_TOTAL = 2000

BRIDGE_BATCH_CLIPS = BATCH_CLIPS
BRIDGE_SHARD_SIZE_CLIPS = SHARD_SIZE_CLIPS

print("Bridge source root:", BRIDGE_RAW_DATASET_ROOT)
print("Bridge output root:", BRIDGE_OUTPUT_ROOT)
print("Bridge runner enabled:", RUN_BRIDGEDATA_V2_PREPROCESS)

In [ ]:
# BridgeData V2 source discovery and inspection.
# Reuses find_tfds_builder_dirs(), get_split_names(), find_video_files(), and find_frame_dirs() from the DROID cells.
import random
from pathlib import Path


def maybe_download_bridge_source():
    if not BRIDGE_DOWNLOAD_DATASET:
        return
    if not BRIDGE_DATASET_LINK:
        raise ValueError("Set BRIDGE_DATASET_LINK before enabling BRIDGE_DOWNLOAD_DATASET.")
    BRIDGE_RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    if str(BRIDGE_DATASET_LINK).startswith("gs://"):
        run(["gsutil", "-m", "cp", "-r", str(BRIDGE_DATASET_LINK), str(BRIDGE_RAW_DOWNLOAD_DIR)])
    else:
        raise ValueError("Only gs:// Bridge downloads are supported by this notebook cell. Add local/Kaggle inputs via BRIDGE_RAW_DATASET_ROOT.")


def bridge_builder_dirs_from_tfds_name():
    if not BRIDGE_TFDS_NAME:
        return []
    builder = tfds.builder(BRIDGE_TFDS_NAME, data_dir=str(BRIDGE_TFDS_DATA_DIR))
    builder_dir = Path(builder.data_dir)
    print("Bridge TFDS builder:", builder.name, builder.version, "dir:", builder_dir)
    if not builder_dir.exists():
        if BRIDGE_ALLOW_TFDS_DOWNLOAD:
            print("Downloading/preparing Bridge TFDS. This is huge and can exceed Kaggle disk quota:", builder_dir)
            builder.download_and_prepare()
        else:
            print("Bridge TFDS is not downloaded. Set BRIDGE_ALLOW_TFDS_DOWNLOAD=True only if you have enough disk,")
            print("or add a pre-downloaded Bridge TFDS/Kaggle Dataset and set BRIDGE_RAW_DATASET_ROOT to it.")
            return []
    return [builder_dir]


def is_gcs_path(path):
    return isinstance(path, str) and path.startswith("gs://")


def find_bridge_builder_dirs():
    roots = []
    if BRIDGE_USE_GCS_STREAMING:
        roots.append(BRIDGE_GCS_BUILDER_DIR)
    search_roots = [BRIDGE_RAW_DATASET_ROOT, BRIDGE_DOWNLOAD_TARGET_ROOT, BRIDGE_RAW_DOWNLOAD_DIR]
    if BRIDGE_SCAN_ALL_KAGGLE_INPUTS:
        search_roots.append(KAGGLE_INPUT_ROOT)
    for root in search_roots:
        if Path(root).exists():
            roots.extend(find_tfds_builder_dirs(root))
    roots.extend(bridge_builder_dirs_from_tfds_name())
    seen = set()
    unique = []
    for root in roots:
        if is_gcs_path(root):
            key = root.rstrip("/")
            value = root
        else:
            root_path = Path(root)
            key = str(root_path.resolve()) if root_path.exists() else str(root_path)
            value = root_path
        if key not in seen:
            seen.add(key)
            unique.append(value)
    return unique


def discover_bridge_rlds_sources(builder_dirs):
    sources = []
    for builder_dir in builder_dirs:
        try:
            builder = tfds.builder_from_directory(str(builder_dir))
            split_names = get_split_names(builder)
            print("Bridge builder:", builder.name, builder.version, "splits:", split_names, "dir:", builder_dir)
            for split in split_names:
                n = builder.info.splits[split].num_examples
                for episode_index in range(n):
                    sources.append(("rlds", {"builder_dir": str(builder_dir), "split": split, "episode_index": episode_index}))
        except Exception as exc:
            print("Skipping invalid TFDS builder dir:", builder_dir, repr(exc))
    return sources


def discover_bridge_plain_sources():
    roots = BRIDGE_RAW_VIDEO_ROOTS or [BRIDGE_RAW_DATASET_ROOT, BRIDGE_DOWNLOAD_TARGET_ROOT]
    sources = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        sources.extend(("video", p) for p in find_video_files(root))
        sources.extend(("frames", p) for p in find_frame_dirs(root))
    return sources


def discover_bridge_sources():
    maybe_download_bridge_source()
    builder_dirs = find_bridge_builder_dirs()
    print("Bridge TFDS builder dirs:")
    for d in builder_dirs[:20]:
        print("-", d)
    sources = discover_bridge_rlds_sources(builder_dirs)
    if not sources:
        print("No Bridge RLDS sources found; scanning videos/frame folders as fallback.")
        sources = discover_bridge_plain_sources()
    random.Random(BRIDGE_RANDOM_SEED).shuffle(sources)
    if BRIDGE_MAX_EPISODES is not None:
        sources = sources[:BRIDGE_MAX_EPISODES]
    print("Bridge sources:", len(sources))
    for item in sources[:10]:
        print(item)
    return sources


def inspect_first_bridge_source(bridge_sources):
    if not bridge_sources:
        print("No Bridge source to inspect yet.")
        return
    kind, payload = bridge_sources[0]
    print("Inspecting first Bridge source:", kind, payload)
    old_camera_keys = CAMERA_KEY_CANDIDATES
    try:
        globals()["CAMERA_KEY_CANDIDATES"] = BRIDGE_CAMERA_KEY_CANDIDATES
        frames = load_source_frames(kind, payload)
        print("Frames loaded:", len(frames))
        if frames:
            print("First frame:", frames[0].shape, frames[0].dtype, int(frames[0].min()), int(frames[0].max()))
            print("Non-overlap clips available:", sum(1 for _ in iter_nonoverlap_windows(frames)))
    finally:
        globals()["CAMERA_KEY_CANDIDATES"] = old_camera_keys


BRIDGE_DISCOVERED_SOURCES = discover_bridge_sources() if RUN_BRIDGEDATA_V2_PREPROCESS else []
if RUN_BRIDGEDATA_V2_PREPROCESS:
    inspect_first_bridge_source(BRIDGE_DISCOVERED_SOURCES)
else:
    print("Bridge discovery skipped. Set RUN_BRIDGEDATA_V2_PREPROCESS=True after configuring the Bridge source.")

In [ ]:
# Convert BridgeData V2 sources to the same Masked-HWM tokenized layout used by 1X/DROID.
# This cell temporarily swaps the globals used by the DROID writer, then restores them.


def run_bridge_preprocess():
    bridge_sources = BRIDGE_DISCOVERED_SOURCES or discover_bridge_sources()
    if not bridge_sources:
        raise RuntimeError(
            "No BridgeData V2 sources found. Configure BRIDGE_RAW_DATASET_ROOT, BRIDGE_DATASET_LINK, "
            "BRIDGE_TFDS_NAME, or BRIDGE_RAW_VIDEO_ROOTS."
        )

    saved = {
        name: globals().get(name)
        for name in [
            "OUTPUT_ROOT", "TRAIN_DIR", "VAL_DIR", "RAW_DATASET_ROOT", "RAW_VIDEO_ROOTS",
            "CAMERA_KEY_CANDIDATES", "WINDOW_SIZE", "TARGET_SIZE", "FRAME_STRIDE",
            "VAL_RATIO", "RANDOM_SEED", "DATASET_HZ", "MAX_EPISODES",
            "MAX_CLIPS_TOTAL", "BATCH_CLIPS", "SHARD_SIZE_CLIPS", "sources",
        ]
    }
    try:
        globals().update({
            "OUTPUT_ROOT": BRIDGE_OUTPUT_ROOT,
            "TRAIN_DIR": BRIDGE_TRAIN_DIR,
            "VAL_DIR": BRIDGE_VAL_DIR,
            "RAW_DATASET_ROOT": BRIDGE_RAW_DATASET_ROOT,
            "RAW_VIDEO_ROOTS": BRIDGE_RAW_VIDEO_ROOTS,
            "CAMERA_KEY_CANDIDATES": BRIDGE_CAMERA_KEY_CANDIDATES,
            "WINDOW_SIZE": BRIDGE_WINDOW_SIZE,
            "TARGET_SIZE": BRIDGE_TARGET_SIZE,
            "FRAME_STRIDE": BRIDGE_FRAME_STRIDE,
            "VAL_RATIO": BRIDGE_VAL_RATIO,
            "RANDOM_SEED": BRIDGE_RANDOM_SEED,
            "DATASET_HZ": BRIDGE_DATASET_HZ,
            "MAX_EPISODES": BRIDGE_MAX_EPISODES,
            "MAX_CLIPS_TOTAL": BRIDGE_MAX_CLIPS_TOTAL,
            "BATCH_CLIPS": BRIDGE_BATCH_CLIPS,
            "SHARD_SIZE_CLIPS": BRIDGE_SHARD_SIZE_CLIPS,
            "sources": bridge_sources,
        })

        _EPISODE_FRAME_CACHE.clear()
        _PRINTED_CAMERA_KEYS.clear()
        prepare_output_dirs()

        val_count = max(1, int(len(bridge_sources) * BRIDGE_VAL_RATIO)) if len(bridge_sources) > 1 else 0
        val_sources = bridge_sources[:val_count]
        train_sources = bridge_sources[val_count:]
        if not train_sources and val_sources:
            train_sources, val_sources = val_sources, []

        print("Bridge train sources:", len(train_sources))
        print("Bridge val sources:", len(val_sources))
        train_stats = write_split("bridge_train", train_sources, BRIDGE_TRAIN_DIR, is_val=False)
        val_stats = write_split("bridge_val", val_sources, BRIDGE_VAL_DIR, is_val=True) if val_sources else (0, 0, 0)
        print("Bridge train stats:", train_stats)
        print("Bridge val stats:", val_stats)
        print("Bridge output root:", BRIDGE_OUTPUT_ROOT)
        return {"train": train_stats, "val": val_stats, "output_root": str(BRIDGE_OUTPUT_ROOT)}
    finally:
        globals().update(saved)


if RUN_BRIDGEDATA_V2_PREPROCESS:
    BRIDGE_PREPROCESS_RESULT = run_bridge_preprocess()
else:
    print("Bridge conversion skipped. Set RUN_BRIDGEDATA_V2_PREPROCESS=True to write", BRIDGE_OUTPUT_ROOT)

In [ ]:
# Verify Bridge output and optionally publish it as a Kaggle Dataset.
# Run after Bridge conversion. Training should use WITH_ACT=False unless a Bridge action adapter is added.
import json
import os
import re
import subprocess
import sys
from pathlib import Path

GENERATED_BRIDGE_ROOT = BRIDGE_OUTPUT_ROOT

if not (GENERATED_BRIDGE_ROOT / "train_v2.0" / "metadata.json").exists():
    print("Bridge tokenized dataset not found yet:", GENERATED_BRIDGE_ROOT)
    print("Run the Bridge conversion cell first.")
else:
    bridge_reports = {}
    for split in ["train", "val"]:
        bridge_reports[split] = inspect_split(GENERATED_BRIDGE_ROOT, split)
        print_split_report("Bridge generated", bridge_reports[split])

    print("\nBridge training readiness")
    bridge_train_ok = bridge_reports["train"]["ok"] and bridge_reports["train"].get("raw_token_dataset_valid_examples_estimate", 0) > 0
    bridge_val_ok = bridge_reports["val"]["ok"] and bridge_reports["val"].get("raw_token_dataset_valid_examples_estimate", 0) > 0
    status_line(bridge_train_ok, "Bridge train split can be read by RawTokenDataset-style loader")
    status_line(bridge_val_ok, "Bridge val split can be read by RawTokenDataset-style loader")
    print("Recommended training config: WITH_ACT=False, TRAIN_DATA_DIR=.../bridge_hwm_tokenized_dataset/train_v2.0, VAL_DATA_DIR=.../val_v2.0")

RUN_BRIDGE_KAGGLE_UPLOAD = False
BRIDGE_KAGGLE_DATASET_SLUG = "bridgedata-v2-hwm-tokenized"
BRIDGE_KAGGLE_DATASET_TITLE = "BridgeData V2 Masked-HWM Cosmos Tokens"
BRIDGE_KAGGLE_DATASET_LICENSE = "CC-BY-4.0"
BRIDGE_MAKE_PRIVATE = True

if RUN_BRIDGE_KAGGLE_UPLOAD:
    if not GENERATED_BRIDGE_ROOT.exists():
        raise FileNotFoundError(f"Bridge output does not exist: {GENERATED_BRIDGE_ROOT}")
    for required in ["train_v2.0/metadata.json", "val_v2.0/metadata.json"]:
        if not (GENERATED_BRIDGE_ROOT / required).exists():
            raise FileNotFoundError(f"Missing required output file: {GENERATED_BRIDGE_ROOT / required}")

    username = require_secret("KAGGLE_USERNAME").strip()
    api_key = require_secret("KAGGLE_KEY").strip()
    if not re.fullmatch(r"[a-z0-9-]+", BRIDGE_KAGGLE_DATASET_SLUG):
        raise ValueError("BRIDGE_KAGGLE_DATASET_SLUG must contain only lowercase letters, numbers, and hyphens.")

    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = api_key
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

    metadata = {
        "id": f"{username}/{BRIDGE_KAGGLE_DATASET_SLUG}",
        "title": BRIDGE_KAGGLE_DATASET_TITLE,
        "licenses": [{"name": BRIDGE_KAGGLE_DATASET_LICENSE}],
        "isPrivate": BRIDGE_MAKE_PRIVATE,
    }
    (GENERATED_BRIDGE_ROOT / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    cmd = ["kaggle", "datasets", "create", "-p", str(GENERATED_BRIDGE_ROOT), "--dir-mode", "zip"]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Kaggle dataset creation failed with exit code {result.returncode}.")
    print("Created Kaggle Dataset:", f"https://www.kaggle.com/datasets/{username}/{BRIDGE_KAGGLE_DATASET_SLUG}")